In [1]:
# Celda 1 - ¿qué GPU me tocó?
!nvidia-smi
!nvcc --version | tail -1

import time
import numpy as np
import cupy as cp

print(cp.cuda.runtime.getDeviceProperties(0)["name"].decode(), "| CuPy", cp.__version__)

Fri Oct  2 01:07:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Celda 2 - medición correcta en GPU
def medir(fn, reps=5, sync=None):
    fn() # calentamiento
    if sync: sync()

    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        if sync: sync()
        ts.append(time.perf_counter() - t0)
    return np.mean(ts), np.std(ts)

sync = cp.cuda.Device().synchronize

for n in [256, 1024, 4096]:
    A = np.random.rand(n, n).astype(np.float32)
    B = np.random.rand(n, n).astype(np.float32)

    Ag, Bg = cp.asarray(A), cp.asarray(B)

    t_cpu, s_cpu = medir(lambda: A @ B)
    t_gpu, s_gpu = medir(lambda: Ag @ Bg, sync=sync)
    t_mal, _ = medir(lambda: Ag @ Bg)

    print(f"n={n:5d} | CPU {t_cpu:.4f} +- {s_cpu:.4f} s | GPU {t_gpu:.5f} +- {s_gpu:.5f} s")
    print(f"          | speedup {t_cpu/t_gpu:7.1f}x GPU | sin sync {t_mal:.6f} s (incorrecto)\n")

n=  256 | CPU 0.0101 +- 0.0028 s | GPU 0.00097 +- 0.00170 s
          | speedup    10.4x GPU | sin sync 0.000068 s (incorrecto)

n= 1024 | CPU 0.0202 +- 0.0001 s | GPU 0.00094 +- 0.00004 s
          | speedup    21.5x GPU | sin sync 0.000058 s (incorrecto)

n= 4096 | CPU 1.4985 +- 0.3723 s | GPU 0.04036 +- 0.00827 s
          | speedup    37.1x GPU | sin sync 0.000071 s (incorrecto)

